# m24 lab · Interpolation, activation maps and isochrones

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

A mapping system knows activation times only at the points it visited; the colored map in between is interpolated. You bound the error of linear interpolation, find the flat spots inverse-distance weighting leaves around each point, interpolate harmonically on a mesh and see what a free boundary does, draw isochrones, and watch interpolation turn a line of block into apparent slow conduction.

| Part | What you do | Equation |
| --- | --- | --- |
| A | Linear interpolation and its error | eq-m24-linear |
| B | Inverse-distance weighting | eq-m24-idw |
| C | Harmonic interpolation on a mesh | eq-m24-harmonic |
| D | Isochrones | eq-m24-isochrone |
| E | Block and coverage | eq-m24-block |

Times in ms, lengths in mm, speeds in mm/ms (= m/s). Conduction velocity 1.0 m/s as in m03; point counts, block delays and fill radii are illustrative. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import interpolation as IP, meshes as M

## Part A · Linear interpolation and its error

On a triangulated surface, linear interpolation inside each triangle uses barycentric weights and is exact for
linear fields. Along a line of points spaced $h$, its error is at most $h^2\max\lvert T''\rvert/8$ (eq-m24-linear). A focal
activation $T = \sqrt{x^2 + y_0^2}/c$ is most curved near the source, so errors concentrate there.

In [ ]:
y0, c = 5.0, 1.0
xf = np.linspace(-30, 30, 6001)
for h in (1.0, 2.0, 5.0, 10.0):
    xd = np.arange(-30, 30 + 1e-9, h)
    err = np.abs(IP.linear_1d(xd, IP.focal_time(xd, y0, c), xf) - IP.focal_time(xf, y0, c))
    print(f"h = {h:4.1f} mm: largest error {err.max():.3f} ms (bound {IP.linear_error_bound(h, 1 / (c * y0)):.3f} ms) at x = {xf[np.argmax(err)]:+.1f} mm")

In [ ]:
# Checks
xd = np.arange(-30, 30 + 1e-9, 5.0)
assert np.abs(IP.linear_1d(xd, IP.focal_time(xd, y0, c), xf) - IP.focal_time(xf, y0, c)).max() <= IP.linear_error_bound(5.0, 1 / (c * y0))

## Part B · Inverse-distance weighting

IDW (m30) averages the data with weights $d^{-p}$ (eq-m24-idw). It never leaves the data range, but for $p > 1$ its
gradient is zero at every data point, so the map has a flat spot around each point (a bull's-eye); for $p = 1$ it has
a cusp there instead. Between points it also flattens the gradient. A conduction velocity computed from that gradient
(m25) reads too fast, and far too fast near the points.

In [ ]:
rng = np.random.default_rng(24)
pts = rng.uniform(0, 40, (60, 2))
src = np.array([12.0, 15.0])
vals = IP.focal_time(pts[:, 0], pts[:, 1], c, *src)
near = pts + 0.2 * rng.normal(size=pts.shape)
far = rng.uniform(5, 35, (300, 2))
for p in (1.0, 2.0, 3.0):
    g_near = np.linalg.norm(IP.idw_gradient(pts, vals, near, p), axis=1)
    g_far = np.linalg.norm(IP.idw_gradient(pts, vals, far, p), axis=1)
    print(f"p = {p}: median |grad T| near points {np.median(g_near):.2f} ms/mm, elsewhere {np.median(g_far):.2f} ms/mm (true 1.00)")

In [ ]:
# Checks
assert np.median(np.linalg.norm(IP.idw_gradient(pts, vals, near, 2.0), axis=1)) < 0.5

## Part C · Harmonic interpolation on a mesh

Solve $L T = 0$ at unknown vertices with $T$ fixed at the mapped ones, using the cotangent Laplacian of m06
(eq-m24-harmonic). With the boundary fixed it reproduces linear fields exactly and never overshoots. Where the mesh has
a free edge, as at a valve or vein opening, the solution behaves as if no activation crossed it, and bends isochrones
to meet the edge at right angles.

In [ ]:
n = 31
V, F = M.lattice_mesh(n, 1.0, jitter=0.2, seed=24)
ii, jj = np.meshgrid(np.arange(n), np.arange(n)); ii, jj = ii.ravel(), jj.ravel()
edge = np.nonzero((ii == 0) | (ii == n - 1) | (jj == 0) | (jj == n - 1))[0]
foc = IP.focal_time(V[:, 0], V[:, 1], c, 15.0, 12.0)
lin = V[:, 0] / c
for k in (30, 80, 200):
    idx = rng.choice(np.setdiff1d(np.arange(len(V)), edge), k, replace=False)
    Th = IP.harmonic(V, F, idx, foc[idx])
    Ti = IP.idw_points(V[idx, :2], foc[idx], V[:, :2], 2.0)
    print(f"{k:3d} points: RMS error harmonic {np.sqrt(np.mean((Th - foc) ** 2)):.2f} ms, IDW {np.sqrt(np.mean((Ti - foc) ** 2)):.2f} ms")
idx = rng.choice(np.setdiff1d(np.arange(len(V)), edge), 60, replace=False)
fixed = np.union1d(idx, edge)
g_free = M.face_gradients(V, F, IP.harmonic(V, F, idx, lin[idx]))
g_fixed = M.face_gradients(V, F, IP.harmonic(V, F, fixed, lin[fixed]))
left = np.isin(F, np.nonzero(ii == 0)[0]).any(1)
print(f"plane wave along x, faces at the left edge: mean dT/dx {g_free[left, 0].mean():.2f} ms/mm with a free edge, "
      f"{g_fixed[left, 0].mean():.2f} with the edge fixed (true 1.00)")

In [ ]:
# Checks
assert abs(g_fixed[left, 0].mean() - 1.0) < 1e-9 and g_free[left, 0].mean() < 0.8

## Part D · Isochrones

An isochrone is a line of equal activation time. On a mesh with linear interpolation it crosses an edge from $i$ to
$j$ at the fraction $(\tau - T_i)/(T_j - T_i)$ (eq-m24-isochrone); isochrones $\Delta\tau$ apart are $c\,\Delta\tau$
apart, so crowding means slow conduction. Draw them for the true focal map and for IDW and harmonic maps from 40
points.

In [ ]:
idx = rng.choice(len(V), 40, replace=False)
maps = {"truth": foc, "harmonic": IP.harmonic(V, F, idx, foc[idx]), "IDW": IP.idw_points(V[idx, :2], foc[idx], V[:, :2], 2.0)}
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (name, T) in zip(ax, maps.items()):
    for tau in np.arange(4, 30, 4):
        for s in IP.isochrone_segments(V, F, T, tau):
            a.plot([s[0][0], s[1][0]], [s[0][1], s[1][1]], "b-", lw=0.8)
    a.plot(V[idx, 0], V[idx, 1], "k.", ms=3); a.set_title(name); a.set_aspect("equal")
plt.show()
segs = IP.isochrone_segments(V, F, lin, 10.0)
print("plane wave at 1.0 m/s: the 10-ms isochrone lies at x =", np.round(np.mean([p[0] for s in segs for p in s]), 6), "mm")

In [ ]:
# Checks
assert np.allclose([p[0] for s in segs for p in s], 10.0)

## Part E · Block and coverage

Interpolation cannot represent a line of block: two points $h$ apart on either side, with activation $\Delta T$ apart,
are joined by a ramp that reads as conduction at $h/\Delta T$ (eq-m24-block). And a map is only colored within a fill
radius $r$ of some point: for points placed at random with density $\rho$, the colored fraction is
$1 - e^{-\rho\pi r^2}$ (eq-m24-block).

In [ ]:
for h, dT in ((4.0, 60.0), (2.0, 60.0), (4.0, 20.0)):
    print(f"spacing {h} mm, activation times {dT:.0f} ms apart: apparent {IP.apparent_velocity_across_block(h, dT):.3f} m/s")
area = 50.0 * 50.0
g = np.stack(np.meshgrid(np.linspace(5, 45, 61), np.linspace(5, 45, 61)), -1).reshape(-1, 2)
for n_pts in (50, 150, 400):
    for r in (3.0, 6.0):
        sim = np.mean([IP.coverage_simulated(rng.uniform(0, 50, (n_pts, 2)), r, g) for _ in range(20)])
        print(f"{n_pts:3d} points, fill radius {r} mm: covered {IP.coverage_fraction(n_pts / area, r):.3f} (simulated {sim:.3f})")

In [ ]:
# Checks
assert abs(IP.coverage_fraction(150 / area, 3.0) - np.mean([IP.coverage_simulated(rng.uniform(0, 50, (150, 2)), 3.0, g) for _ in range(20)])) < 0.03

## Where to go next

- Add the annotation noise of m22 to the point times and see which interpolant amplifies it.
- Put a block line in the 2D patch and compare isochrone crowding from block with true slow conduction.
- m25 turns these maps into conduction velocities; m26 asks how many points a map needs.